<a href="https://colab.research.google.com/github/Bootcamp-DA-P3/Proyecto2-Maria_y_Carol/blob/main/NotebookMovies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [97]:
# Importar librerías y montar Google Drive
import json
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

# Guardar los csv en una carpeta llamada 'proyecto_datos' en Drive
movies = pd.read_csv('/content/drive/MyDrive/proyecto_datos/tmdb_5000_movies.csv')
credits = pd.read_csv('/content/drive/MyDrive/proyecto_datos/tmdb_5000_credits.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [98]:
# Rutas a los archivos
ruta_movies = '/content/drive/MyDrive/proyecto_datos/tmdb_5000_movies.csv'
ruta_credits = '/content/drive/MyDrive/proyecto_datos/tmdb_5000_credits.csv'

# Cargar los datos
movies = pd.read_csv(ruta_movies)
credits = pd.read_csv(ruta_credits)

print("Datos cargados correctamente.")
print(f"Tamaño de movies: {movies.shape}")
print(f"Tamaño de credits: {credits.shape}")

Datos cargados correctamente.
Tamaño de movies: (4803, 20)
Tamaño de credits: (4803, 4)


In [99]:
# Eliminar la columna 'title' de credits para evitar duplicados al unir
credits = credits.drop(columns=['title'])

# Unir los DataFrames
# Usar 'id' de movies y 'movie_id' de credits
# Eliminar 'movie_id' porque es lo mismo que la columna 'id'
df = movies.merge(credits, left_on='id', right_on='movie_id').drop(columns=['movie_id'])

# 'homepage' tiene muchísimos nulos y no sirve para analizar
# 'tagline' es la frase publicitaria, irrelevante para datos numéricos
# 'status' casi todas son "Released", no aporta varianza
columnas_a_borrar = ['homepage', 'tagline', 'status']
df = df.drop(columnas_a_borrar, axis=1)

print("Columnas innecesarias eliminadas.")

Columnas innecesarias eliminadas.


In [100]:
# Eliminar las filas que no tienen fecha de estreno
df = df.dropna(subset=['release_date'])

# Convertir la columna de texto a formato fecha (datetime) de Pandas
df['release_date'] = pd.to_datetime(df['release_date'])

# Extraer el año a una nueva columna
df['release_year'] = df['release_date'].dt.year

print("Fechas limpiadas y columna de 'Año' creada.")

Fechas limpiadas y columna de 'Año' creada.


In [101]:
# Guardar este DataFrame limpio en Drive
ruta_limpia = '/content/drive/MyDrive/proyecto_datos/tmdb_estructurado_parte1.csv'
df.to_csv(ruta_limpia, index=False)

print(f"Archivo guardado y listo: {ruta_limpia}")

Archivo guardado y listo: /content/drive/MyDrive/proyecto_datos/tmdb_estructurado_parte1.csv


In [102]:
# Cargar el dataset de la Parte 1
ruta_entrada = '/content/drive/MyDrive/proyecto_datos/tmdb_estructurado_parte1.csv'
df = pd.read_csv(ruta_entrada)

In [103]:
# Función para procesar JSONs y extraer 'name'
def extraer_nombres_json(cadena):
    if pd.isna(cadena):
        return []
    try:
        # Se reemplazan las comillas simples (') por comillas dobles (") para que json.loads() lo procese sin error
        cadena_formateada = str(cadena).replace("'", '"')
        datos = json.loads(cadena_formateada)
        return [elem['name'] for elem in datos if isinstance(elem, dict) and 'name' in elem]
    except Exception:
        return []

In [104]:
# Función para extraer los 3 actores principales de 'cast'
def extraer_top_actores_json(cadena, top=3):
    if pd.isna(cadena):
        return []
    try:
        cadena_formateada = str(cadena).replace("'", '"')
        datos = json.loads(cadena_formateada)
        return [elem['name'] for elem in datos[:top] if isinstance(elem, dict) and 'name' in elem]
    except Exception:
        return []

In [105]:
# Función para extraer al Director de 'crew'
def extraer_director_json(cadena):
    if pd.isna(cadena):
        return None
    try:
        cadena_formateada = str(cadena).replace("'", '"')
        datos = json.loads(cadena_formateada)
        for elem in datos:
            if isinstance(elem, dict) and elem.get('job') == 'Director':
                return elem['name']
        return None
    except Exception:
        return None

In [106]:
# Aplicar la conversión a las columnas del DataFrame
df['genres_list'] = df['genres'].apply(extraer_nombres_json)
df['keywords_list'] = df['keywords'].apply(extraer_nombres_json)
df['top_cast'] = df['cast'].apply(extraer_top_actores_json)
df['director'] = df['crew'].apply(extraer_director_json)

# Eliminar las columnas originales en formato texto/JSON no procesadas
cols_json_raw = ['genres', 'keywords', 'cast', 'crew', 'production_companies', 'production_countries', 'spoken_languages']
df_final = df.drop(columns=cols_json_raw)

# Guardar el dataset limpio definitivo
ruta_salida = '/content/drive/MyDrive/proyecto_datos/tmdb_limpio_final.csv'
df_final.to_csv(ruta_salida, index=False)

print(f"Procesamiento con la librería json completado con éxito. Archivo final: {ruta_salida}")

Procesamiento con la librería json completado con éxito. Archivo final: /content/drive/MyDrive/proyecto_datos/tmdb_limpio_final.csv
